# Generate Missing Embeddings**Runs on Colab or local Jupyter.** Recommended on Colab (free GPU) for the two open-source models, since they need GPU inference to finish in reasonable time across the full corpus. If a local GPU is available this runs there too; if not, it still runs on CPU, just slower. This is detected automatically below, no manual switch needed.**Purpose of this notebook**Before the paradigm benchmark (pgvector vs MongoDB vs Qdrant) can begin, we need to pick which embedding model to standardize on. That decision requires comparing candidate models on retrieval quality first. This notebook fills the embedding gaps needed for that model comparison:| Model | abstracts | pdf_chunks ||---|---|---|| `text-embedding-3-small` | done | done || `text-embedding-3-large` | done | **missing — generated here** || `BAAI/bge-small-en-v1.5` | **missing — generated here** | **missing — generated here** || `Alibaba-NLP/gte-base-en-v1.5` | **missing — generated here** | **missing — generated here** |**Why the open-source models want a GPU**: they need to run local inference over the full corpus (5 categories x however many chunks/pages each). `text-embedding-3-large` is an API call so it needs no GPU either way; it's included here so all generation lives in one notebook and one run leaves everything the evaluation notebook needs, ready in one place.**Why we don't touch `text-embedding-3-small`**: it's already complete for both abstracts and pdf_chunks, regenerating it would burn API budget for zero new information.**Output convention**: every model writes to `data/embeddings/{model_name}/{abstracts|pdf_chunks}/{category}.jsonl`, one JSON object per line: `{"chunk_id": ..., "embedding": [...]}`. This exactly matches the existing `text-embedding-3-small` files, so the evaluation notebook's loader doesn't need per-model special-casing.**Source text**: pulled from the HF dataset `Sakhiur/empirical-rag-paradigm-benchmark`, `abstracts/by_category/` and `pdf_chunks/` subfolders. This notebook only reads text from HF and writes new embeddings back to HF; it never touches the raw corpus files.

## SetupWorks on Colab or local Jupyter. `IN_COLAB` (detected below) controls the couple of places behavior has to differ: secrets handling and GPU runtime messaging. Package installation is the same command either way; pip is idempotent about already-satisfied requirements, so re-running this cell on a machine that already has these packages is harmless.

In [ ]:
# sentence-transformers covers both open-source models with one API.# huggingface_hub for pushing the finished jsonl files back to the private dataset repo.# python-dotenv is optional (only used locally, see the secrets cell below); harmless# to install even on Colab where it won't be used.%pip install -q sentence-transformers openai huggingface_hub python-dotenv

In [ ]:
import osimport jsonimport timefrom pathlib import Pathdef running_in_colab() -> bool:    try:        import google.colab  # noqa: F401        return True    except ImportError:        return FalseIN_COLAB = running_in_colab()print(f"Environment: {'Colab' if IN_COLAB else 'local Jupyter'}")

### API keys and HF tokenOn Colab, secrets come from the Colab Secrets panel (key icon in the left sidebar) — this avoids ever typing a key into a cell. Locally, the same two environment variables (`HF_TOKEN`, `OPENAI_API_KEY`) are expected to already be set in the shell that launched Jupyter, or in a `.env` file loaded by `python-dotenv`. Either path lands in the same place: both variables present in `os.environ` before the next cell runs.

In [ ]:
if IN_COLAB:    from google.colab import userdata    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")else:    # Local: try a .env file first (via python-dotenv), then fall back to whatever    # is already in the shell environment. Never prompts or hardcodes a key here.    try:        from dotenv import load_dotenv        load_dotenv()    except ImportError:        pass    missing = [k for k in ("HF_TOKEN", "OPENAI_API_KEY") if not os.environ.get(k)]    if missing:        raise RuntimeError(            f"Missing required environment variable(s): {missing}. "            "Set them in your shell (export HF_TOKEN=...) or in a .env file "            "in the notebook's working directory before re-running this cell."        )print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## ConfigurationCentral place to control which models and content types get generated this run. Kept as a single cell so a partial run (e.g. only the open-source models, or only pdf_chunks) is a one-line edit rather than hunting through the notebook.

In [ ]:
HF_REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"# Sakhiur's five categories. Swap this list if running for Sadman's categories instead.CATEGORIES = ["cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE"]CONTENT_TYPES = ["abstracts", "pdf_chunks"]# Which (model, content_type) combinations actually need generating.# Explicit rather than "generate everything" so re-running after a partial failure# doesn't waste OpenAI budget or GPU time on work that's already done.GENERATION_PLAN = {    "text-embedding-3-large": ["pdf_chunks"],          # abstracts already exist    "BAAI/bge-small-en-v1.5": ["abstracts", "pdf_chunks"],    "Alibaba-NLP/gte-base-en-v1.5": ["abstracts", "pdf_chunks"],}LOCAL_OUTPUT_ROOT = Path("data/embeddings")

## Load source text from Hugging FaceWe need the raw text to embed, not just chunk IDs. Abstracts and pdf_chunks are stored as separate HF dataset subsets, each keyed by the same `chunk_id` convention the embeddings use (bare arXiv ID for abstracts, `{arxiv_id}_p{page}` for pdf_chunks). Loading once per content type and reusing across all three models in the plan avoids redundant HF downloads.

In [ ]:
from datasets import load_datasetdef load_source_texts(content_type: str, category: str) -> dict[str, str]:    """Returns {chunk_id: text} for one category and content type.    The mapping to HF subset/config names may need adjusting once you confirm    the exact dataset schema; this assumes a config per category under each    top-level content type, consistent with how the corpus was harvested.    """    if content_type == "abstracts":        subset = f"abstracts/by_category/{category}"    else:        subset = f"pdf_chunks/{category}"    ds = load_dataset(HF_REPO_ID, data_files=f"{subset}.jsonl", split="train")    text_field = "abstract" if content_type == "abstracts" else "text"    return {row["chunk_id"]: row[text_field] for row in ds}

## Skip-existing resumabilityGeneration over the full corpus with an open-source model on a single Colab GPU can take a while, and Colab sessions can disconnect. Checking which chunk_ids are already written lets a re-run pick up where it left off instead of restarting from scratch. This mirrors the skip-existing pattern already used elsewhere in the pipeline.

In [ ]:
def load_existing_ids(output_path: Path) -> set[str]:    if not output_path.exists():        return set()    existing = set()    with open(output_path, "r") as f:        for line in f:            line = line.strip()            if not line:                continue            existing.add(json.loads(line)["chunk_id"])    return existingdef append_embeddings(output_path: Path, records: list[dict]) -> None:    output_path.parent.mkdir(parents=True, exist_ok=True)    with open(output_path, "a") as f:        for record in records:            f.write(json.dumps(record) + "\n")

## Embedding functionsOne function per model family. Open-source models are batched on GPU for throughput; `text-embedding-3-large` goes through the OpenAI API with basic retry-on-failure, since API calls can transiently fail and we don't want one hiccup to kill a multi-hour run.

In [ ]:
import torchfrom sentence_transformers import SentenceTransformerdevice = "cuda" if torch.cuda.is_available() else "cpu"print(f"Using device: {device}")if device == "cpu":    if IN_COLAB:        print("No GPU detected. Runtime > Change runtime type > GPU, then re-run this cell.")    else:        print(            "No GPU detected on this machine. Generation will still work on CPU, "            "just slower for the open-source models. To use a GPU instead, either "            "run this notebook on a machine with a CUDA-capable GPU + the matching "            "torch build installed, or run it on Colab (Runtime > Change runtime type > GPU)."        )

In [ ]:
_st_model_cache: dict[str, SentenceTransformer] = {}def embed_with_sentence_transformers(    model_name: str, chunk_ids: list[str], texts: list[str], batch_size: int = 64) -> list[dict]:    if model_name not in _st_model_cache:        _st_model_cache[model_name] = SentenceTransformer(model_name, device=device)    model = _st_model_cache[model_name]    # normalize_embeddings=True makes the vectors unit-length, which is what both    # BGE and GTE expect for cosine-similarity retrieval and keeps this consistent    # with how OpenAI embeddings are typically consumed downstream.    vectors = model.encode(        texts,        batch_size=batch_size,        show_progress_bar=True,        normalize_embeddings=True,        convert_to_numpy=True,    )    return [        {"chunk_id": cid, "embedding": vec.tolist()}        for cid, vec in zip(chunk_ids, vectors)    ]

In [ ]:
from openai import OpenAIopenai_client = OpenAI()def embed_with_openai(    model_name: str,    chunk_ids: list[str],    texts: list[str],    batch_size: int = 100,    max_retries: int = 3,) -> list[dict]:    results = []    for start in range(0, len(texts), batch_size):        batch_ids = chunk_ids[start:start + batch_size]        batch_texts = texts[start:start + batch_size]        for attempt in range(max_retries):            try:                response = openai_client.embeddings.create(                    model=model_name, input=batch_texts                )                break            except Exception as e:                if attempt == max_retries - 1:                    raise                wait = 2 ** attempt                print(f"  retry {attempt + 1}/{max_retries} after error: {e} (waiting {wait}s)")                time.sleep(wait)        for cid, item in zip(batch_ids, response.data):            results.append({"chunk_id": cid, "embedding": item.embedding})    return results

## Run generationLoops over the generation plan: for each (model, content_type) pair, for each category, load source text, skip chunk_ids already embedded, embed the rest, append to the jsonl file. Structured this way so a crash partway through only loses the current in-flight batch, not prior progress.

In [ ]:
def is_openai_model(model_name: str) -> bool:    return model_name.startswith("text-embedding-")for model_name, content_types in GENERATION_PLAN.items():    safe_model_name = model_name  # used as a path segment; already filesystem-safe    for content_type in content_types:        print(f"\n=== {model_name} | {content_type} ===")        for category in CATEGORIES:            output_path = LOCAL_OUTPUT_ROOT / safe_model_name / content_type / f"{category}.jsonl"            existing_ids = load_existing_ids(output_path)            source_texts = load_source_texts(content_type, category)            todo = {                cid: text for cid, text in source_texts.items() if cid not in existing_ids            }            if not todo:                print(f"  {category}: {len(source_texts)} chunks, all already embedded, skipping")                continue            print(f"  {category}: {len(todo)} of {len(source_texts)} chunks need embedding")            chunk_ids = list(todo.keys())            texts = list(todo.values())            if is_openai_model(model_name):                records = embed_with_openai(model_name, chunk_ids, texts)            else:                records = embed_with_sentence_transformers(model_name, chunk_ids, texts)            append_embeddings(output_path, records)            print(f"  {category}: wrote {len(records)} embeddings to {output_path}")

## Verify outputQuick sanity check before uploading: confirm every planned (model, content_type, category) combination produced a file with the expected number of records, and that embedding dimensions are consistent within a model. Catching a truncated or malformed file here is much cheaper than discovering it mid-evaluation.

In [ ]:
for model_name, content_types in GENERATION_PLAN.items():    for content_type in content_types:        for category in CATEGORIES:            output_path = LOCAL_OUTPUT_ROOT / model_name / content_type / f"{category}.jsonl"            if not output_path.exists():                print(f"MISSING: {output_path}")                continue            dims = set()            count = 0            with open(output_path, "r") as f:                for line in f:                    record = json.loads(line)                    dims.add(len(record["embedding"]))                    count += 1            dim_str = dims.pop() if len(dims) == 1 else f"INCONSISTENT: {dims}"            print(f"{output_path}: {count} records, dim={dim_str}")

## Upload to Hugging FacePushes the newly generated embeddings to the private dataset repo alongside the existing `text-embedding-3-small` files, so the evaluation notebook (run locally or on any machine) can pull everything from one place rather than needing these files copied around manually.

In [ ]:
from huggingface_hub import HfApiapi = HfApi()for model_name, content_types in GENERATION_PLAN.items():    for content_type in content_types:        for category in CATEGORIES:            local_path = LOCAL_OUTPUT_ROOT / model_name / content_type / f"{category}.jsonl"            if not local_path.exists():                continue            repo_path = f"embeddings/{model_name}/{content_type}/{category}.jsonl"            api.upload_file(                path_or_fileobj=str(local_path),                path_in_repo=repo_path,                repo_id=HF_REPO_ID,                repo_type="dataset",            )            print(f"uploaded {repo_path}")

## Next stepOnce this notebook completes and all files pass the verification check above, move to the evaluation notebook. That notebook pulls everything (queries CSVs + all four models' embeddings) from `Sakhiur/empirical-rag-paradigm-benchmark` and runs the actual model comparison. No GPU needed there since it's cosine similarity and ranking, so it can run on a local machine.